# Regression ML Practice 

Using ML techniques instead of traditional statistics techniques

## SETUP

In [1]:
import warnings

import datasets
import pandas as pd

import sklearn.feature_selection as skft
import sklearn.impute as skimp
import sklearn.linear_model as sklin
import sklearn.pipeline as skpipe
import sklearn.preprocessing as skprep
import sklearn.metrics as skmetrics
import sklearn.model_selection as skm

In [2]:
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

## DATA COLLECTION

In [3]:
cols_with_known_gaps = [
    "Alley",
    "MasVnrType",
    "FireplaceQu",
    "PoolQC",
    "Fence",
    "MiscFeature",
]

In [4]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ds = datasets.load_dataset(
        "michaelmallari/house-prices-advanced-regression-techniques"
    )

src_df = ds["train"].to_pandas().drop(cols_with_known_gaps, axis=1)

In [5]:
dependent_var = "SalePrice"

## DATA PREPROCESSING

In [6]:
train_df, test_df = skm.train_test_split(src_df, test_size=0.2)

y_train_s, y_test_s = train_df.loc[:, dependent_var], test_df.loc[:, dependent_var]
X_train_df, X_test_df = (
    train_df.drop(dependent_var, axis=1),
    test_df.drop(dependent_var, axis=1),
)

In [7]:
pipe = skpipe.Pipeline(
    [
        (
            "encoder",
            skprep.OneHotEncoder(
                sparse_output=False, drop="first", handle_unknown="infrequent_if_exist"
            ),
        ),
        ("imputer", skimp.SimpleImputer(missing_values=pd.NA)),
        ("selector", skft.SelectPercentile(score_func=skft.f_regression)),
        ("polynomial", skprep.PolynomialFeatures()),
        ("scaler", skprep.StandardScaler()),
        ("regressor", sklin.Lasso()),
    ]
)

hyperparams = {
    # {'imputer__strategy': 'most_frequent',
    #  'polynomial__degree': 2,
    #  'regressor': Lasso(),
    #  'regressor__alpha': 7.5,
    #  'selector__percentile': 10}
    "imputer__strategy": ["mean", "median", "most_frequent"],
    "selector__percentile": [10, 50, 75],
    "polynomial__degree": [1, 2, 3],
    "regressor": [
        sklin.Lasso(),
        sklin.Ridge(),
        sklin.QuantileRegressor(),
    ],
    "regressor__alpha": [1.0, 2.5, 5.0, 7.5, 10.0],
}

hyperparams_precise = {
    "imputer__strategy": ["most_frequent"],
    "selector__percentile": [13, 14, 15, 16, 17, 18, 19],
    "polynomial__degree": [2],
    "regressor__alpha": [7.8, 7.9, 8, 8.1, 8.2],
}

## MODEL TUNING

In [14]:
results = skm.GridSearchCV(
    pipe,
    hyperparams_precise,
    cv=5,
    n_jobs=4,
    verbose=2,
    scoring="neg_root_mean_squared_error",
).fit(X_train_df, y_train_s)

In [9]:
results.best_params_

{'imputer__strategy': 'most_frequent',
 'polynomial__degree': 2,
 'regressor__alpha': 8.2,
 'selector__percentile': 17}

## MODEL TEST

In [15]:
predictions = results.predict(X_test_df)

In [11]:
skmetrics.r2_score(y_test_s, predictions)

0.7828652245424914

In [12]:
skmetrics.root_mean_squared_error(y_test_s, predictions)

34761.82375971379

In [13]:
skmetrics.mean_absolute_error(y_test_s, predictions)

22799.694238229175